# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrahmanshaheen1/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages are, on average, longer and younger than declining pages. It compares pages with rising impressions against pages with falling impressions and finds that the growing group averages about 3.2K words and 184 days of age, while the declining group averages about 2.3K words and 230 days of age.

**Methodology question:**  
How exactly is the "growing" versus "declining" label constructed, and is the comparison using information from the same time window as the outcome? If the label is based on recent impression change, then the result supports an observed association between age, depth, and current momentum, but it does not by itself show that making a page longer or refreshing it will cause future growth. I would also ask whether the same pattern remains when controlling for client, topic, and search visibility.

**Safe interpretation:**  
The portfolio shows a measured directional association: growing pages were younger and longer on average. This is useful for generating review hypotheses, but it is not causal proof that age or word count creates growth.


### Finding 2 — AI Model Performance

The paper compares OpenAI- and Gemini-associated content inside age-controlled publication cohorts. It reports that each provider family leads in some age windows, so the paper treats the result as exploratory rather than claiming one provider universally performs better.

**Methodology question:**  
Is controlling for age sufficient to support a provider-performance claim? I would also want to know whether topic mix, client, editing process, rollout timing, content intent, and other production differences are balanced between the OpenAI and Gemini cohorts. If those factors differ systematically, they may explain part of the measured performance gap.

**Safe interpretation:**  
The age-controlled comparison suggests that provider-family performance varies across cohorts. It supports a directional process comparison, but it does not establish that one AI provider causes better SEO performance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Check Colab Secrets."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

APRIL_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

feature_frame = con.sql(f"""
    WITH march_features AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0))
                AS march_impressions,

            SUM(COALESCE(gsc_clicks, 0))
                AS march_clicks,

            ROUND(
                100.0 *
                SUM(COALESCE(gsc_clicks, 0)) /
                NULLIF(
                    SUM(COALESCE(gsc_impressions, 0)),
                    0
                ),
                4
            ) AS march_ctr_pct,

            AVG(
                CASE
                    WHEN gsc_impressions > 0
                         AND gsc_avg_position > 0
                    THEN gsc_avg_position
                END
            ) AS march_avg_position,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0
                    THEN report_date
                END
            ) AS march_active_days

        FROM {MARCH_DAILY}

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING
            SUM(COALESCE(gsc_impressions, 0)) >= 100
    ),

    april_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0))
                AS april_impressions,

            COUNT(DISTINCT report_date)
                AS april_observed_days

        FROM {APRIL_DAILY}

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.*,
        a.april_impressions,

        CASE
            WHEN a.april_impressions
                 < 0.80 * m.march_impressions
            THEN 1
            ELSE 0
        END AS is_next_month_decline

    FROM march_features AS m

    INNER JOIN april_outcomes AS a
        USING (
            client_hash_id,
            content_hash_id
        )

    WHERE a.april_observed_days > 0
""").df()

FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_days",
]

print(f"Rows: {len(feature_frame):,}")
print(f"Clients: {feature_frame['client_hash_id'].nunique():,}")
print(
    "Decline rate: "
    f"{feature_frame['is_next_month_decline'].mean():.1%}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101,441
Clients: 44
Decline rate: 51.7%


In [2]:
def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    actual_k = min(k, len(y_array))

    top_indices = np.argsort(
        score_array
    )[-actual_k:]

    return y_array[top_indices].mean()


def make_logistic_model():
    return Pipeline(
        steps=[
            (
                "missing_values",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    )

In [3]:
# =========================================================
# BEFORE — random page-level split
# =========================================================

random_train, random_test = train_test_split(
    feature_frame,
    test_size=0.25,
    random_state=42,
    stratify=feature_frame["is_next_month_decline"],
)

random_model = make_logistic_model()

random_model.fit(
    random_train[FEATURES],
    random_train["is_next_month_decline"],
)

random_scores = random_model.predict_proba(
    random_test[FEATURES]
)[:, 1]

random_p20 = precision_at_k(
    random_test["is_next_month_decline"],
    random_scores,
    20,
)

random_p50 = precision_at_k(
    random_test["is_next_month_decline"],
    random_scores,
    50,
)

random_client_overlap = len(
    set(random_train["client_hash_id"])
    & set(random_test["client_hash_id"])
)

print("BEFORE — Random page split")
print(f"Precision@20: {random_p20:.3f}")
print(f"Precision@50: {random_p50:.3f}")
print(
    f"Clients appearing in BOTH train and test: "
    f"{random_client_overlap:,}"
)

BEFORE — Random page split
Precision@20: 0.700
Precision@50: 0.740
Clients appearing in BOTH train and test: 42


In [4]:
# =========================================================
# AFTER — grouped split by client
# =========================================================

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        feature_frame,
        groups=feature_frame["client_hash_id"],
    )
)

group_train = feature_frame.iloc[
    group_train_idx
].copy()

group_test = feature_frame.iloc[
    group_test_idx
].copy()

group_model = make_logistic_model()

group_model.fit(
    group_train[FEATURES],
    group_train["is_next_month_decline"],
)

group_scores = group_model.predict_proba(
    group_test[FEATURES]
)[:, 1]

group_p20 = precision_at_k(
    group_test["is_next_month_decline"],
    group_scores,
    20,
)

group_p50 = precision_at_k(
    group_test["is_next_month_decline"],
    group_scores,
    50,
)

group_client_overlap = len(
    set(group_train["client_hash_id"])
    & set(group_test["client_hash_id"])
)

print("AFTER — Grouped client split")
print(f"Precision@20: {group_p20:.3f}")
print(f"Precision@50: {group_p50:.3f}")
print(
    f"Clients appearing in BOTH train and test: "
    f"{group_client_overlap:,}"
)

AFTER — Grouped client split
Precision@20: 0.850
Precision@50: 0.700
Clients appearing in BOTH train and test: 0


In [5]:
validation_comparison = pd.DataFrame(
    {
        "Validation design": [
            "Random page split (before)",
            "Grouped client split (after)",
        ],
        "Precision@20": [
            random_p20,
            group_p20,
        ],
        "Precision@50": [
            random_p50,
            group_p50,
        ],
        "Client overlap": [
            random_client_overlap,
            group_client_overlap,
        ],
    }
)

validation_comparison[
    ["Precision@20", "Precision@50"]
] = validation_comparison[
    ["Precision@20", "Precision@50"]
].round(3)

validation_comparison

,Validation design,Precision@20,Precision@50,Client overlap
0,Random page split (before),0.70,0.74,42
1,Grouped client split (after),0.85,0.70,0


### Validation design interpretation

The random page-level split produced Precision@20 = 0.700 and Precision@50 = 0.740. However, 42 clients appeared in both the training and test sets. This means the model was often evaluated on pages belonging to clients whose other pages it had already seen during training.

The grouped-client split removed this overlap completely. Precision@20 was 0.850 and Precision@50 was 0.700, with zero clients appearing in both train and test.

The grouped result is the more defensible estimate for this project because it tests the model on pages from entirely unseen clients. The lower Precision@50 compared with the random split may indicate that some page-level performance patterns are client-specific and do not transfer perfectly to new clients.

The validation change does not prove that the random split was invalid for every purpose, but for the intended decision-support use case, grouped validation provides a stronger test of generalization.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
leakage_audit = pd.DataFrame(
    [
        {
            "field": "march_impressions",
            "role": "feature",
            "available_at_decision_time": True,
            "future_window": False,
            "label_derived": False,
            "verdict": "SAFE",
        },
        {
            "field": "march_clicks",
            "role": "feature",
            "available_at_decision_time": True,
            "future_window": False,
            "label_derived": False,
            "verdict": "SAFE",
        },
        {
            "field": "march_ctr_pct",
            "role": "feature",
            "available_at_decision_time": True,
            "future_window": False,
            "label_derived": False,
            "verdict": "SAFE",
        },
        {
            "field": "march_avg_position",
            "role": "feature",
            "available_at_decision_time": True,
            "future_window": False,
            "label_derived": False,
            "verdict": "SAFE",
        },
        {
            "field": "march_active_days",
            "role": "feature",
            "available_at_decision_time": True,
            "future_window": False,
            "label_derived": False,
            "verdict": "SAFE",
        },
        {
            "field": "april_impressions",
            "role": "label source only",
            "available_at_decision_time": False,
            "future_window": True,
            "label_derived": False,
            "verdict": "EXCLUDED",
        },
        {
            "field": "is_next_month_decline",
            "role": "target",
            "available_at_decision_time": False,
            "future_window": True,
            "label_derived": True,
            "verdict": "TARGET ONLY",
        },
    ]
)

leakage_audit

,field,role,available_at_decision_time,future_window,label_derived,verdict
0,march_impressions,feature,True,False,False,SAFE
1,march_clicks,feature,True,False,False,SAFE
2,march_ctr_pct,feature,True,False,False,SAFE
3,march_avg_position,feature,True,False,False,SAFE
4,march_active_days,feature,True,False,False,SAFE
5,april_impressions,label source only,False,True,False,EXCLUDED
6,is_next_month_decline,target,False,True,True,TARGET ONLY


In [10]:
unsafe_features = leakage_audit[
    (leakage_audit["role"] == "feature")
    & (
        (~leakage_audit["available_at_decision_time"])
        | leakage_audit["future_window"]
        | leakage_audit["label_derived"]
    )
]

print(f"Unsafe model features found: {len(unsafe_features)}")

assert len(unsafe_features) == 0

print(
    "Leakage audit passed: all final model features "
    "are available by the March decision moment "
    "and contain no future or label-derived information."
)

Unsafe model features found: 0
Leakage audit passed: all final model features are available by the March decision moment and contain no future or label-derived information.


In [11]:
print("High-scoring negatives — model ranks them highly, but no April decline was observed:")
display(false_positives)

print("\nLow-scoring positives — April decline occurred despite a low model score:")
display(false_negatives)

High-scoring negatives — model ranks them highly, but no April decline was observed:


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_active_days,is_next_month_decline,model_score
27429,client_3f0ce4d44fe94f3d,content_e3a63930f7422987,447.0,0.0,0.0,1.926054,31,0,0.632474
27397,client_3f0ce4d44fe94f3d,content_f40387e3d7a349cd,1378.0,0.0,0.0,3.284776,31,0,0.629828
27680,client_3f0ce4d44fe94f3d,content_e8d5c1a8cc08cf39,947.0,0.0,0.0,3.378749,31,0,0.629748
27441,client_3f0ce4d44fe94f3d,content_671db9022e987d77,930.0,0.0,0.0,3.623242,31,0,0.629309
27538,client_3f0ce4d44fe94f3d,content_615d616041741df9,345.0,0.0,0.0,4.461645,31,0,0.627913



Low-scoring positives — April decline occurred despite a low model score:


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_active_days,is_next_month_decline,model_score
9589,client_f623b01661d4bfe4,content_b96495c6b2911b12,160.0,12.0,7.5000,39.523844,28,1,0.001318
60197,client_f623b01661d4bfe4,content_0a124186f945b8cd,259.0,19.0,7.3359,20.279238,31,1,0.002079
60177,client_f623b01661d4bfe4,content_cdbf00ce4999800f,123.0,7.0,5.6911,28.980395,26,1,0.006197
9555,client_f623b01661d4bfe4,content_7b50820e76b1a006,2243.0,103.0,4.5921,31.888335,31,1,0.009260
60188,client_f623b01661d4bfe4,content_d957f0f71b23e023,124.0,6.0,4.8387,48.113639,27,1,0.012168


### Leakage and failure audit

The final model uses only five March features that were available before the April outcome window. `april_impressions` is retained only as a source for constructing the future target, and `is_next_month_decline` is the target itself. Neither enters the model feature set. The automated audit found zero unsafe model features.

The failure examples show that the model is still imperfect even without leakage. Some high-scoring pages did not decline in April, while some pages that received very low scores did decline. These examples are useful because they show where the five March signals do not fully capture future behavior.

False positives may occur when weak March click or CTR signals look risky but the page later remains stable. False negatives may occur when a page looks healthy in March but declines in April because of changes that are not represented by the current feature set.

These examples support using the model as decision-support rather than an automatic content-action system.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Earlier, too-broad claim:**  
"Logistic Regression performed best."

**Rewritten claim:**  
On the grouped-client holdout used in this experiment, Logistic Regression achieved the strongest measured ranking performance among the three methods tested: Precision@20 = 0.850 and Precision@50 = 0.700, compared with 0.550 / 0.600 for the Week-4 baseline and 0.800 / 0.640 for Random Forest.

This result supports using Logistic Regression as the current decision-support model for this experiment. It does not establish that Logistic Regression will outperform other models on every client, time period, feature set, or future dataset.

The grouped validation result is the result I retain because the test set contains entirely unseen clients. The random page split produced Precision@50 = 0.740, but 42 clients appeared in both training and testing, so that number is a weaker estimate of performance on new clients.

The model's outputs should be interpreted as measured ranking signals that support human review, not as causal evidence or guaranteed predictions of future content performance.

## Self-check

Before you submit, confirm each line honestly:

- [✔ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [ ✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.